# Week 1: ARX — Next-Day Binary Classification

**Goal:** Predict whether the wheat futures price moves **up (1)** or **down (0)**
on the next trading day, using an ARX-style logistic regression classifier.

**Pipeline (per assignment specification):**

1. Load raw FRED-MD vintage (`2025-10-MD.csv`) and apply **t-code stationarity
   transformations** to each of the 31 macro variables.
2. Account for the **one-month reporting delay** (shift dates forward by one month).
3. **Forward-fill** monthly macro data to daily frequency.
4. Input features: **30 lagged daily closing prices** + **31 transformed macro variables**.
5. ARX regressor matrix: AR price lags + exogenous macro lags → logistic regression.
6. **StandardScaler fit on training data only** per CV fold.
7. **5-fold expanding-window time-series cross-validation**.
8. Grid search over `ar_lags`, `exog_lags`, `alpha`.
9. Metrics: Accuracy, Precision, Recall, F1, AUC-ROC, Confusion Matrix.

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path
import warnings
import pickle

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from itertools import product

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print("All imports loaded.")

In [ ]:
class BaseForecastModel(ABC):
    """Simple base class for forecasting models."""

    def __init__(self, task_type: str, **hyperparameters):
        self.task_type = task_type
        self.hyperparameters = hyperparameters

    @abstractmethod
    def fit(self, X_train, y_train):
        pass

    @abstractmethod
    def predict(self, X):
        pass

    @abstractmethod
    def evaluate(self, X_test, y_test):
        pass

    @abstractmethod
    def save(self, filepath: str):
        pass

    @abstractmethod
    def load(self, filepath: str):
        pass

print("BaseForecastModel defined.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/Quants ")
INVESTING_DIR = BASE_DIR / "Investing.com"

WHEAT_2018 = INVESTING_DIR / "US Wheat Futures Historical Data_2018.csv"
WHEAT_2025 = INVESTING_DIR / "US Wheat Futures Historical Data_2025.csv"


def load_price_csv(path: Path) -> pd.Series:
    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").set_index("Date")
    df["Price"] = df["Price"].astype(str).str.replace(",", "", regex=False).astype(float)
    return df["Price"]


def load_and_merge_wheat_prices(path_2018: Path, path_2025: Path) -> pd.Series:
    p1 = load_price_csv(path_2018)
    p2 = load_price_csv(path_2025)
    merged = pd.concat([p1, p2.iloc[1:]], axis=0)
    merged = merged[~merged.index.duplicated(keep="last")].sort_index()
    merged.name = "Price"
    return merged


df_price = load_and_merge_wheat_prices(WHEAT_2018, WHEAT_2025)
print(f"Wheat prices: {len(df_price)} days, "
      f"{df_price.index.min().date()} to {df_price.index.max().date()}")

## Load FRED-MD and apply t-code stationarity transformations

| T-Code | Transformation |
|--------|----------------|
| 1 | Level (no change) |
| 2 | First difference |
| 3 | Second difference |
| 4 | Log |
| 5 | Log first difference |
| 6 | Log second difference |
| 7 | Percent change first difference |

In [ ]:
fred_md_path = '/content/drive/MyDrive/Quants /FredMD_Dataset/2025-10-MD.csv'

tcodes_row = pd.read_csv(fred_md_path, nrows=1)
fred_raw = pd.read_csv(fred_md_path, skiprows=[1])
fred_raw['sasdate'] = pd.to_datetime(fred_raw['sasdate'], format='%m/%d/%Y')
fred_raw = fred_raw.set_index('sasdate').sort_index()

features_31 = [
    "RPI", "W875RX1", "CMRMTSPLx", "IPFPNSS", "USWTRADE",
    "USTRADE", "BUSLOANS", "CONSPI", "S&P 500", "S&P PE ratio",
    "FEDFUNDS", "TB3MS", "TB6MS", "GS1", "GS5", "GS10",
    "AAA", "BAA", "TB3SMFFM", "TB6SMFFM", "T1YFFM",
    "T5YFFM", "T10YFFM", "AAAFFM", "BAAFFM",
    "EXSZUSx", "EXJPUSx", "EXUSUKx", "EXCAUSx",
    "PPICMM", "UMCSENTx",
]

tcode_map = {}
for col in features_31:
    tcode_map[col] = int(float(tcodes_row[col].values[0]))

print("T-codes for the 31 required variables:")
for col, tc in tcode_map.items():
    print(f"  {col:20s} -> t-code {tc}")

In [ ]:
def apply_tcode(series, tcode):
    if tcode == 1:
        return series
    elif tcode == 2:
        return series.diff()
    elif tcode == 3:
        return series.diff().diff()
    elif tcode == 4:
        return np.log(series.clip(lower=1e-10))
    elif tcode == 5:
        return np.log(series.clip(lower=1e-10)).diff()
    elif tcode == 6:
        return np.log(series.clip(lower=1e-10)).diff().diff()
    elif tcode == 7:
        return (series / series.shift(1) - 1).diff()
    else:
        return series

fred_selected = fred_raw[features_31].copy()
fred_selected = fred_selected.apply(pd.to_numeric, errors='coerce')

for col in features_31:
    fred_selected[col] = apply_tcode(fred_selected[col], tcode_map[col])

fred_transformed = fred_selected.dropna()
fred_transformed = fred_transformed.replace([np.inf, -np.inf], np.nan).dropna()

print(f"After t-code transformation: {fred_transformed.shape}")
print(f"Date range: {fred_transformed.index.min().date()} to {fred_transformed.index.max().date()}")

In [ ]:
df_macro = fred_transformed.copy()
df_macro.index = df_macro.index + pd.DateOffset(months=1)

all_days = pd.date_range(
    df_macro.index.min(),
    df_macro.index.max() + pd.offsets.MonthEnd(0),
    freq='D',
)
df_macro = df_macro.reindex(all_days).ffill()

print(f"Daily macro features: {df_macro.shape}")

In [ ]:
df_cls = df_macro.join(df_price, how='inner')
df_cls = df_cls['2008-01-01':]

cols_order = [c for c in df_cls.columns if c != 'Price'] + ['Price']
df_cls = df_cls[cols_order]

print(f"Classification dataset: {df_cls.shape}")
print(f"Date range: {df_cls.index.min().date()} to {df_cls.index.max().date()}")

In [ ]:
lookback_cls = 30
price_col_idx = df_cls.columns.get_loc('Price')
data_cls = df_cls.values

X_cls, y_cls = [], []
for i in range(lookback_cls, len(data_cls)):
    X_cls.append(data_cls[i - lookback_cls : i])
    y_cls.append(1 if data_cls[i, price_col_idx] > data_cls[i - 1, price_col_idx] else 0)

X_cls = np.array(X_cls)
y_cls = np.array(y_cls)

print(f"X shape: {X_cls.shape}  |  y shape: {y_cls.shape}")
print(f"Class distribution: Up={y_cls.sum()} ({y_cls.mean():.2%}), "
      f"Down={len(y_cls)-y_cls.sum()} ({1-y_cls.mean():.2%})")

In [ ]:
split = int(len(X_cls) * 0.8)
X_train, y_train = X_cls[:split], y_cls[:split]
X_test,  y_test  = X_cls[split:], y_cls[split:]

dates_cls = df_cls.index[lookback_cls:]
train_dates, test_dates = dates_cls[:split], dates_cls[split:]

print(f"Train: {X_train.shape[0]} ({train_dates[0].date()} to {train_dates[-1].date()})")
print(f"Test:  {X_test.shape[0]} ({test_dates[0].date()} to {test_dates[-1].date()})")
print(f"Train dist: Up={y_train.sum()} ({y_train.mean():.2%}), Down={len(y_train)-y_train.sum()}")
print(f"Test  dist: Up={y_test.sum()} ({y_test.mean():.2%}),  Down={len(y_test)-y_test.sum()}")

## ARX Classifier

Same regressor-matrix construction as regression ARX:
- **AR lags**: last `ar_lags` price values from the lookback window.
- **Exogenous lags**: last `exog_lags` days of 31 macro features, flattened.
- Classification via **logistic regression** with balanced class weights.

In [ ]:
class ARXClassifier(BaseForecastModel):
    """ARX-style binary classifier using logistic regression."""

    def __init__(self, task_type='classification', ar_lags=5,
                 exog_lags=3, alpha=1.0, n_exog_features=31):
        super().__init__(task_type=task_type, ar_lags=ar_lags,
                         exog_lags=exog_lags, alpha=alpha,
                         n_exog_features=n_exog_features)
        self.ar_lags = ar_lags
        self.exog_lags = exog_lags
        self.alpha = alpha
        self.n_exog_features = n_exog_features
        self.clf_ = None
        self.x_scaler_ = None

    def _regressor_matrix(self, X_s):
        price_lags = X_s[:, -self.ar_lags:, -1]
        exog_lags = X_s[:, -self.exog_lags:, :self.n_exog_features]
        return np.concatenate([price_lags, exog_lags.reshape(X_s.shape[0], -1)], axis=1)

    def _scale(self, X, fit=False):
        nf = X.shape[2]
        if fit:
            self.x_scaler_ = StandardScaler()
            flat = self.x_scaler_.fit_transform(X.reshape(-1, nf))
        else:
            flat = self.x_scaler_.transform(X.reshape(-1, nf))
        return flat.reshape(X.shape)

    def fit(self, X_train, y_train):
        X_s = self._scale(X_train, fit=True)
        Z = self._regressor_matrix(X_s)
        C = 1.0 / max(self.alpha, 1e-8)
        self.clf_ = LogisticRegression(
            C=C, class_weight='balanced', max_iter=1000,
            solver='lbfgs', random_state=42,
        )
        self.clf_.fit(Z, y_train)
        preds = self.clf_.predict(Z)
        print(f"  Train Acc: {accuracy_score(y_train, preds):.4f}  "
              f"F1: {f1_score(y_train, preds, zero_division=0):.4f}")
        return self

    def predict(self, X):
        X_s = self._scale(X)
        return self.clf_.predict_proba(self._regressor_matrix(X_s))[:, 1]

    def evaluate(self, X_test, y_test, threshold=0.5):
        probs = self.predict(X_test)
        preds = (probs >= threshold).astype(int)
        m = {
            'accuracy':  accuracy_score(y_test, preds),
            'precision': precision_score(y_test, preds, zero_division=0),
            'recall':    recall_score(y_test, preds, zero_division=0),
            'f1':        f1_score(y_test, preds, zero_division=0),
        }
        try: m['auc_roc'] = roc_auc_score(y_test, probs)
        except ValueError: m['auc_roc'] = 0.5
        return m

    def save(self, filepath):
        with open(filepath, 'wb') as f:
            pickle.dump({'clf': self.clf_, 'scaler': self.x_scaler_,
                         'hp': self.hyperparameters}, f)

    def load(self, filepath):
        with open(filepath, 'rb') as f:
            d = pickle.load(f)
        self.clf_ = d['clf']
        self.x_scaler_ = d['scaler']
        hp = d['hp']
        self.hyperparameters = hp
        self.ar_lags = hp['ar_lags']
        self.exog_lags = hp['exog_lags']
        self.alpha = hp['alpha']
        self.n_exog_features = hp['n_exog_features']
        return self

print("ARXClassifier defined.")

In [ ]:
def cv_arx_cls(X_train, y_train, ar_lags=5, exog_lags=3, alpha=1.0):
    """5-fold expanding-window CV for ARX classification."""
    from sklearn.metrics import log_loss
    tscv = TimeSeriesSplit(n_splits=5)
    n_exog = X_train.shape[2] - 1
    fold_m = {k: [] for k in ['accuracy','precision','recall','f1','auc_roc','val_loss']}
    all_probs, all_true = [], []

    for fold, (ti, vi) in enumerate(tscv.split(X_train)):
        m = ARXClassifier(ar_lags=ar_lags, exog_lags=exog_lags,
                          alpha=alpha, n_exog_features=n_exog)
        Xs = m._scale(X_train[ti], fit=True)
        Z_tr = m._regressor_matrix(Xs)
        C = 1.0 / max(alpha, 1e-8)
        m.clf_ = LogisticRegression(C=C, class_weight='balanced',
                                    max_iter=1000, solver='lbfgs', random_state=42)
        m.clf_.fit(Z_tr, y_train[ti])

        Xv = m._scale(X_train[vi])
        Z_v = m._regressor_matrix(Xv)
        probs = m.clf_.predict_proba(Z_v)[:, 1]
        preds = (probs >= 0.5).astype(int)
        yv = y_train[vi]

        fold_m['val_loss'].append(log_loss(yv, probs))
        fold_m['accuracy'].append(accuracy_score(yv, preds))
        fold_m['precision'].append(precision_score(yv, preds, zero_division=0))
        fold_m['recall'].append(recall_score(yv, preds, zero_division=0))
        fold_m['f1'].append(f1_score(yv, preds, zero_division=0))
        try: fold_m['auc_roc'].append(roc_auc_score(yv, probs))
        except ValueError: fold_m['auc_roc'].append(0.5)

        all_probs.extend(probs); all_true.extend(yv)
        print(f"  Fold {fold+1}: Acc={fold_m['accuracy'][-1]:.4f}  "
              f"F1={fold_m['f1'][-1]:.4f}  AUC={fold_m['auc_roc'][-1]:.4f}")

    return {k: np.mean(v) for k,v in fold_m.items()}, np.array(all_true), np.array(all_probs)

## Default Hyperparameters (`ar_lags=5, exog_lags=3, alpha=1.0`)

In [ ]:
default_m, def_true, def_probs = cv_arx_cls(X_train, y_train)

print("\n=== Default CV Metrics (5-fold avg) ===")
for k, v in default_m.items():
    print(f"  {k:12s}: {v:.6f}")

def_preds = (def_probs >= 0.5).astype(int)
print("\n" + classification_report(def_true, def_preds, target_names=['Down','Up']))

## Grid Search

Sweep `ar_lags`, `exog_lags`, `alpha` — maximise average F1.

In [ ]:
grid = {
    'ar_lags':   [3, 5, 7, 10],
    'exog_lags': [1, 3, 5],
    'alpha':     [0.01, 0.1, 1.0, 5.0],
}

results = []
for ar, ex, al in tqdm(list(product(grid['ar_lags'], grid['exog_lags'], grid['alpha'])),
                       desc="Grid search"):
    avg, _, _ = cv_arx_cls(X_train, y_train, ar_lags=ar, exog_lags=ex, alpha=al)
    results.append({'ar_lags': ar, 'exog_lags': ex, 'alpha': al,
                    **{f'cv_{k}': v for k, v in avg.items()}})

res_df = pd.DataFrame(results).sort_values('cv_f1', ascending=False).reset_index(drop=True)
print("Top 5 by F1:")
display(res_df.head(5))

best = res_df.iloc[0]
best_ar, best_ex, best_al = int(best['ar_lags']), int(best['exog_lags']), float(best['alpha'])
print(f"\nBest: ar_lags={best_ar}, exog_lags={best_ex}, alpha={best_al}, F1={best['cv_f1']:.6f}")

In [ ]:
grid_m, grid_true, grid_probs = cv_arx_cls(
    X_train, y_train, ar_lags=best_ar, exog_lags=best_ex, alpha=best_al,
)
print("\n=== Grid Search CV Metrics (5-fold avg) ===")
for k, v in grid_m.items():
    print(f"  {k:12s}: {v:.6f}")

grid_preds = (grid_probs >= 0.5).astype(int)
print("\n" + classification_report(grid_true, grid_preds, target_names=['Down','Up']))

## Final Test Set Evaluation

In [ ]:
final = ARXClassifier(ar_lags=best_ar, exog_lags=best_ex,
                      alpha=best_al, n_exog_features=31)
final.fit(X_train, y_train)

test_probs = final.predict(X_test)
test_preds = (test_probs >= 0.5).astype(int)

print("\n=== Test Set Results ===")
print(f"  Accuracy:  {accuracy_score(y_test, test_preds):.6f}")
print(f"  Precision: {precision_score(y_test, test_preds, zero_division=0):.6f}")
print(f"  Recall:    {recall_score(y_test, test_preds, zero_division=0):.6f}")
print(f"  F1:        {f1_score(y_test, test_preds, zero_division=0):.6f}")
try:
    print(f"  AUC-ROC:   {roc_auc_score(y_test, test_probs):.6f}")
except ValueError:
    print("  AUC-ROC:   N/A")

print("\n" + classification_report(y_test, test_preds, target_names=['Down','Up']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

cm = confusion_matrix(y_test, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Down','Up'], yticklabels=['Down','Up'], ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Test — Confusion Matrix')

try:
    fpr, tpr, _ = roc_curve(y_test, test_probs)
    auc = roc_auc_score(y_test, test_probs)
    axes[1].plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {auc:.4f}')
    axes[1].plot([0,1],[0,1], 'navy', lw=1, ls='--', label='Random')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].set_title('Test — ROC Curve'); axes[1].legend()
except ValueError:
    axes[1].text(0.5, 0.5, 'ROC unavailable', ha='center')

plt.tight_layout()
plt.show()

## Model Comparison

In [ ]:
def _auc(yt, yp):
    try: return roc_auc_score(yt, yp)
    except ValueError: return 0.5

comp = pd.DataFrame([
    {'Model': 'ARX Default CV',     **{k: default_m[k] for k in ['accuracy','precision','recall','f1','auc_roc']}},
    {'Model': 'ARX Grid Search CV', **{k: grid_m[k]    for k in ['accuracy','precision','recall','f1','auc_roc']}},
    {'Model': 'ARX Final Test',
     'accuracy': accuracy_score(y_test, test_preds),
     'precision': precision_score(y_test, test_preds, zero_division=0),
     'recall': recall_score(y_test, test_preds, zero_division=0),
     'f1': f1_score(y_test, test_preds, zero_division=0),
     'auc_roc': _auc(y_test, test_probs)},
]).set_index('Model')

comp.style.highlight_max(subset=['accuracy','f1','auc_roc'], color='#d4edda').format('{:.6f}')